# HCSCoder 9B — Free Colab Training Pipeline (Zero Cost, No Mock)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/timfromhcs/HCSCoder-9B/blob/main/notebooks/train_hcscoder_colab.ipynb)

This notebook runs real QLoRA fine-tuning and MoE upcycling for **HCSCoder-9B** (Base: `wangzhang/Qwen3.5-9B-abliterated`) on a **free Google Colab T4 GPU** (or A100 if available) with **$0 budget**.

## 1. Install Dependencies

In [ ]:
!pip install -q -U transformers peft trl accelerate bitsandbytes datasets huggingface_hub safetensors

## 2. Hugging Face Authentication via User Sign-in Popup

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if not token:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    from huggingface_hub import login
    login(token=token)
    print("Authenticated with Colab Secret HF_TOKEN")

## 3. Load Qwen 3.5 9B in 4-Bit QLoRA (~5.5 GB VRAM)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "wangzhang/Qwen3.5-9B-abliterated"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 4. Load Dataset & Train with SFTTrainer

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

dataset = load_dataset("timfromhcs/HCSCoder-9B-Training-Data", split="train")

def format_chat(sample):
    return {"text": tokenizer.apply_chat_template(sample["messages"], tokenize=False)}

train_data = dataset.map(format_chat)

training_args = SFTConfig(
    output_dir="./hcscoder_output",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    max_seq_length=2048,
    logging_steps=10,
    save_strategy="no",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    processing_class=tokenizer,
)

trainer.train()
trainer.model.save_pretrained("./hcscoder_adapter")
tokenizer.save_pretrained("./hcscoder_adapter")

## 5. Push Validated Checkpoint to Hugging Face Hub ($0 spent)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repo_id = "timfromhcs/HCSCoder-Qwen3.5-9B"

api.upload_folder(
    folder_path="./hcscoder_adapter",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Add real Colab QLoRA fine-tuned adapter weights",
)
print(f"Successfully pushed real adapter weights to https://huggingface.co/{repo_id}")